# AI-Driven Emotion Detection Model Training Pipeline (Kaggle)

This notebook demonstrates how to load, clean, preprocess, and train the deep learning models for the student emotion detection task.

**Emotions Classes:** Bored, Confident, Confused, Curious, Frustrated

**Models:**
1. Bidirectional LSTM (BiLSTM) in PyTorch
2. Fine-tuned BERT model (prajjwal1/bert-small) using PyTorch

In [ ]:
# Install dependencies if not already present on Kaggle
!pip install -q transformers scikit-learn pandas numpy torch plotly matplotlib

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import json
import re
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running training on: {device}")

## 1. Load and Preprocess Data

In [ ]:
# Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.strip()

# Sample dataset loading (replace with Kaggle input path if needed)
data_path = "../data/student_emotions_dataset.csv"
if not Path(data_path).exists():
    # Generate dummy data for verification on Kaggle if running standalone
    print("Dataset path not found, generating sample dataset...")
    dummy_data = []
    for label in ["Bored", "Confident", "Confused", "Curious", "Frustrated"]:
        for i in range(200):
            dummy_data.append({"text": f"This is a sample query representing {label} emotion validation step {i}", "emotion": label, "field": "Computer Science"})
    df = pd.DataFrame(dummy_data)
else:
    df = pd.read_csv(data_path)

df["cleaned_text"] = df["text"].apply(clean_text)
print(df.head())
print("Emotion distribution:\n", df["emotion"].value_counts())

## 2. Train-Test Split & Vocab Building

In [ ]:
EMOTION_MAP = {"Bored": 0, "Confident": 1, "Confused": 2, "Curious": 3, "Frustrated": 4}
INV_EMOTION_MAP = {v: k for k, v in EMOTION_MAP.items()}

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["emotion"])

# Vocab for BiLSTM
vocab = {"<PAD>": 0, "<UNK>": 1}
for text in train_df["cleaned_text"]:
    for word in text.split():
        if word not in vocab:
            vocab[word] = len(vocab)
print(f"BiLSTM Vocab Size: {len(vocab)}")

## 3. BiLSTM Training Pipeline

In [ ]:
class StudentEmotionDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=80):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        words = clean_text(text).split()
        indices = [self.vocab.get(w, 1) for w in words]
        if len(indices) > self.max_len:
            indices = indices[:self.max_len]
        else:
            indices = indices + [0] * (self.max_len - len(indices))
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label, dtype=torch.long)

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=5, num_layers=2):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, bidirectional=True, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        out = torch.mean(lstm_out, dim=1)
        out = self.fc(self.dropout(out))
        return out

In [ ]:
train_loader = DataLoader(StudentEmotionDataset(train_df["text"].values, train_df["emotion"].map(EMOTION_MAP).values, vocab), batch_size=64, shuffle=True)
val_loader = DataLoader(StudentEmotionDataset(val_df["text"].values, val_df["emotion"].map(EMOTION_MAP).values, vocab), batch_size=64, shuffle=False)

bilstm_model = BiLSTMClassifier(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(bilstm_model.parameters(), lr=0.001)

for epoch in range(5):
    bilstm_model.train()
    t_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(bilstm_model(x), y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * x.size(0)
    print(f"BiLSTM Epoch {epoch+1} | Loss: {t_loss/len(train_loader.dataset):.4f}")

## 4. BERT Sequence Classification training (Hugging Face)

In [ ]:
class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=80):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(clean_text(text), max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

tokenizer = AutoTokenizer.from_pretrained("prajjwal1/bert-small")
bert_model = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-small", num_labels=5).to(device)

train_bert_loader = DataLoader(BERTDataset(train_df["text"].values, train_df["emotion"].map(EMOTION_MAP).values, tokenizer), batch_size=32, shuffle=True)
val_bert_loader = DataLoader(BERTDataset(val_df["text"].values, val_df["emotion"].map(EMOTION_MAP).values, tokenizer), batch_size=32, shuffle=False)

optimizer_bert = optim.AdamW(bert_model.parameters(), lr=3e-5)
for epoch in range(2):
    bert_model.train()
    loss_val = 0.0
    for batch in train_bert_loader:
        optimizer_bert.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer_bert.step()
        loss_val += outputs.loss.item() * input_ids.size(0)
    print(f"BERT Epoch {epoch+1} | Loss: {loss_val/len(train_bert_loader.dataset):.4f}")

## 5. Model Export

In [ ]:
# Export scripts
print("Exporting trained model parameters...")
torch.save(bilstm_model.state_dict(), "bilstm_model.pt")
bert_model.save_pretrained("./bert_model")
tokenizer.save_pretrained("./bert_model")
print("All artifacts ready for integration!")